In [3]:
# setup cv
import os
os.environ["MUJOCO_GL"] = "glfw"  # or "egl"

# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# robots
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun, log_rerun_data
from lerobot.record import record_loop

# env
from lerobot.envs.utils import env_to_policy_features

# torch
from torch import cuda

# utils
from dotenv import load_dotenv
from IPython.display import clear_output
import numpy as np
import time

# my code
from src.utils import scroll_print
from src.env_rollout import env_rollout
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR
from envs.so101_env_utils import ACTIONS, DT, JOINTS, MUJOCO_DIR, JOINTS_MAX, JOINTS_MIN, SO101OBSTYPES, SO101TASKS

# my env
from envs.so101_env_config import SO101EnvConfig, make_so101_env
from src.robot_utils import sweep_leader_dataset, norm_modes_to_ranges

# set up os_env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

ImportError: cannot import name 'ConvertToLeRobotObservation' from 'lerobot.scripts.rl.gym_manipulator' (C:\Github\IDC_Project-Sim_Real_Sim\libs\lerobot\src\lerobot\scripts\rl\gym_manipulator.py)

Configuration:

In [40]:
RECORD_TASK      = False
NUM_EPISODES     = 1
PUSH_TO_HUB      = False
TASK_DESCRIPTION = "assemble peg into hole"
REPO_NAME        = 'so101_table_leg_assemble'

### Connect Leader

In [2]:
teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

teleop = SO101Leader(teleop_config)

# get motor ranges
leader_ranges = norm_modes_to_ranges(teleop)
print('Joint space range for the actual robot:')
for name, (lo, hi) in zip(ACTIONS, leader_ranges):
    print(f"  {name:>12}: [{lo:.1f}, {hi:.1f}]")

NameError: name 'norm_modes_to_ranges' is not defined

In [68]:
try:
    teleop.connect()
    print("Teleop device connected successfully.")
except Exception as e:
    print(e)

Teleop device connected successfully.


### Build Env

In [71]:
env_cfg = SO101EnvConfig(
    task                  = "TableLegAssembleTask",
    obs_type              = "pixels_agent_pos",
    device                = "cpu",
    external_joint_ranges = leader_ranges,
    control_time_s        = 60
    )
env = make_so101_env(env_cfg, torch_actions = True, lerobot_obs=True)
scroll_print(env_cfg)

### Build Dataset

In [63]:
if RECORD_TASK:
    # from lerobot.envs.utils import env_to_policy_features
    # env_to_policy_features(env_cfg)
    # TODO handle datasets
    dataset.utils.build_dataset_frame

    action_features = hw_to_dataset_features(robot.action_features, "action")
    obs_features = hw_to_dataset_features(robot.observation_features, "observation")
    dataset_features = {**action_features, **obs_features}
    pprint(dataset_features)

    dataset_path = DATASETS_DIR / REPO_NAME
    if dataset_path.exists():
        dataset=LeRobotDataset(
            repo_id=f"{HF_NAME}/{REPO_NAME}",
            root=f"{DATASETS_DIR}\\{REPO_NAME}"
        )
    else:
        dataset = LeRobotDataset.create(
            repo_id=f"{HF_NAME}/{REPO_NAME}",
            root=f"{DATASETS_DIR}\\{REPO_NAME}",
            fps=FPS,
            features=dataset_features,
            robot_type=robot.name,
            use_videos=True,
            image_writer_threads=4,
        )

### Teleop Loop

In [64]:
# synthetic_actions = sweep_leader_dataset(leader_ranges, 100, 20)
# synthetic_actions = np.zeros_like(synthetic_actions)

In [73]:
env_rollout(
    display_rerun = True,
    env_cfg       = env_cfg,
    env           = env,
    num_episodes  = NUM_EPISODES,
    teleop        = teleop,
    dataset       = None,
)

Escape key pressed. Stopping data recording...Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...

Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...
Episode 1/1: reward=11.75, success=False


{'episodes': [{'reward': 11.75, 'done': False, 'success': False}],
 'avg_reward': np.float64(11.75),
 'success_rate': 0.0}

In [66]:
teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()

Escape key pressed. Stopping data recording...Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...

Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...
Escape key pressed. Stopping data recording...
